# IMDB Dataset Text Classification using RNN

In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Input, SimpleRNN, Dense
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
import keras_tuner as kt

In [2]:
df = pd.read_csv("../datasets/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [5]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['review'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42
)


In [6]:
vocab_size = 10000
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train_raw)


X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_test_seq = tokenizer.texts_to_sequences(X_test_raw)

In [7]:
max_length = 200
X_train_padded = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

In [8]:
embedding_dim = 64

model = Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    SimpleRNN(units=64, activation='relu', return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', 
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 650,369 (2.48 MB)

 Trainable params: 650,369 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
history = model.fit(
    X_train_padded, y_train,
    validation_data=(X_test_padded, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop]
)

In [11]:
# Using keras_tuner to improve model accuracy
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(max_length,)))
    embedding_dim = hp.Choice('embedding_dim', values=[32, 64, 128])
    model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim))
    rnn_type = hp.Choice('rnn_type', values=['SimpleRNN'])
    rnn_units = hp.Int('rnn_units', min_value=32, max_value=128, step=32)
    model.add(SimpleRNN(units=rnn_units, activation='relu'))
    dense_units = hp.Int('dense_units', min_value=16, max_value=64, step=16)
    model.add(Dense(units=dense_units, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [12]:
stop_early = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=6,             # Adjust depending on desired search breadth
    executions_per_trial=1,
    directory='kt_imdb_dir',
    project_name='rnn_tuning',
    overwrite=True
)

In [13]:
tuner.search_space_summary()

Search space summary
Default search space size: 5
embedding_dim (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64, 128], 'ordered': True}
rnn_type (Choice)
{'default': 'SimpleRNN', 'conditions': [], 'values': ['SimpleRNN'], 'ordered': False}
rnn_units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 128, 'step': 32, 'sampling': 'linear'}
dense_units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 16, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [14]:
tuner.search(
    X_train_padded, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_padded, y_test),
    callbacks=[stop_early]
)

Trial 6 Complete [00h 03m 01s]
val_accuracy: 0.5112000107765198

Best val_accuracy So Far: 0.8409000039100647
Total elapsed time: 00h 13m 29s


In [16]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:")
print(f" - Embedding Dim: {best_hps.get('embedding_dim')}")
print(f" - RNN Type: {best_hps.get('rnn_type')}")
print(f" - RNN Units: {best_hps.get('rnn_units')}")
print(f" - Dropout: {best_hps.get('dropout')}")
print(f" - Dense Units: {best_hps.get('dense_units')}")
print(f" - Learning Rate: {best_hps.get('learning_rate')}")

Best Hyperparameters:
 - Embedding Dim: 64
 - RNN Type: SimpleRNN
 - RNN Units: 64


KeyError: 'dropout does not exist.'

In [17]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()


/Users/vedanshdwivedi/ai_course/venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(store)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 48)             │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 651,425 (2.48 MB)

 Trainable params: 651,425 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
loss, acc = best_model.evaluate(X_test_padded, y_test)
print(f"Tuned Model Test Accuracy: {acc:.2%}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8409 - loss: 0.3951
Tuned Model Test Accuracy: 84.09%


In [21]:
best_model.save("imdb_review_classification.keras")

In [20]:
# Prediction
def predict_sentiment(review_text: str, model=model, tokenizer=tokenizer, max_length=200) -> dict:
    sequences = tokenizer.texts_to_sequences([review_text])
    padded_seq = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    prediction = model.predict(padded_seq, verbose=0)
    score = float(prediction[0][0])
    label = "Positive" if score >= 0.5 else "Negative"
    return {
        "review": review_text,
        "sentiment": label,
        "probability": score
    }

In [22]:
predict_sentiment("This movie was absolutely breathtaking. I did not realise how two hours passed by. Amazing.", best_model, tokenizer)

{'review': 'This movie was absolutely breathtaking. I did not realise how two hours passed by. Amazing.',
 'sentiment': 'Positive',
 'probability': 0.852175235748291}

In [23]:
import pickle

with open("tokenizer.pickle", "wb") as file:
    pickle.dump(tokenizer, file)